In [18]:
!python3.10 -m pip install python-Levenshtein

Defaulting to user installation because normal site-packages is not writeable

[notice] A new release of pip is available: 24.1.2 -> 24.2
[notice] To update, run: pip3.10 install --upgrade pip


In [19]:
from typing import Tuple, List, Optional
import os
import glob
import pickle
import copy

import pandas as pd
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity, euclidean_distances
from tqdm import tqdm
import Levenshtein

# import torch


In [20]:
# Paths
DATA_PATH = '../../../my-coles-gnn-experimetns/scenario_gender/data/'
SORTED_NEIGHBOURS_DIR_PATH = os.path.join(DATA_PATH, 'sorted_neighbours')

In [21]:
def get_embeddings_list(embedding_paths: List[str]) -> List[Tuple[np.ndarray, np.ndarray]]:
    embeddings_list = []
    for embedding_path in tqdm(embedding_paths):
        with open(embedding_path, 'rb') as f:
            embeddings_pd = pickle.load(f)
        try:
            embeddings_list.append(convert_pd_embeddings_to_np(embeddings_pd))
        except:
            print(embedding_path)
    return embeddings_list


def convert_pd_embeddings_to_np(embeddings_pd: pd.DataFrame, 
                                customer_id_col: str = 'customer_id'
                                ) -> Tuple[np.ndarray, np.ndarray]:
    ### Preconditions Contract
    assert customer_id_col == embeddings_pd.columns[0], f"First column expected to be {customer_id_col}. Got {embeddings_pd.columns[0]}"
    n_digits_in_emb_shape = len(str(len(embeddings_pd.columns) - 2))
    for i, column_name in enumerate(embeddings_pd.columns[1:]):
        expected_col_name = f'emb_{str(i).zfill(n_digits_in_emb_shape)}'
        assert column_name == expected_col_name, f"Expected {expected_col_name}. Got {column_name}"
    ###

    ids = embeddings_pd[customer_id_col].to_numpy()
    embeddings = embeddings_pd.drop(columns=[customer_id_col]).to_numpy()

    return ids, embeddings


# Abstract similarity function
def compute_similarity(embeddings, method='cosine'):
    if method == 'cosine':
        # Cosine similarity (higher is more similar)
        similarity_matrix = cosine_similarity(embeddings)
    elif method == 'euclidean':
        # Euclidean distance (lower is more similar)
        similarity_matrix = -euclidean_distances(embeddings)
    elif method == 'dot':
        # Dot product (higher is more similar)
        similarity_matrix = np.dot(embeddings, embeddings.T)
    else:
        raise ValueError(f"Unknown method: {method}")
    
    return similarity_matrix


def get_sorted_neighbours_from_similarity_matrix(similarity_matrix):
    res = []
    n = len(similarity_matrix)
    for i in range(n):
        # Sort indices of neighbors by similarity (excluding the point itself)
        neighbors = np.argsort(-similarity_matrix[i, :])
        neighbors = neighbors[neighbors != i]
        res.append(neighbors)
    return res


def precision_at_k(neighbors, labels, k=5):
    n = len(labels)
    precisions = []
    
    for i in range(n):
        # Check top K neighbors
        top_k_neighbors = neighbors[i][:k]
        same_label_count = sum(labels[top_k_neighbors] == labels[i])
        
        # Precision for this point
        precisions.append(same_label_count / k)
    
    # Average precision across all points
    return np.mean(precisions)


def average_difference_between_neighbor_lists(list_1, list_2, k=10):
    """
    Calculate the average difference between two lists of top-k neighbors for each point.
    
    Parameters:
    - list_1: 2D numpy array (n_points x k) where each row contains indices of top-k neighbors for a point.
    - list_2: 2D numpy array (n_points x k) similar to list_1.
    - k: Integer, the number of top neighbors considered.
    
    Returns:
    - avg_difference: The average difference between the two lists of neighbors.
    """
    n_points = len(list_1)
    total_difference = 0
    
    for i in range(n_points):
        list1_slice = list_1[i][:k]
        list2_slice = list_2[i][:k]
        # Compute the difference for the i-th point
        # difference = len(set(list1_slice) - set(list2_slice)) + len(set(list2_slice) - set(list1_slice))
        
        difference = len(set(list1_slice) - set(list2_slice))
        # Sum up the total difference
        total_difference += difference / k
    
    # Calculate the average difference
    avg_difference = total_difference / n_points
    
    return avg_difference


def average_Levinstein_distance(list_1, list_2, k:Optional[int]=None):
    """
    Calculate the average Levenshtein distance between two lists of top-k neighbors for each point.
    
    Parameters:
    - list_1: 2D numpy array (n_points x k) where each row contains indices of top-k neighbors for a point.
    - list_2: 2D numpy array (n_points x k) similar to list_1.
    - k: Integer, the number of top neighbors considered.
    
    Returns:
    - avg_Levinstein_distance: The average Levenshtein distance between the two lists of neighbors.
    """
    if k is None:
        k = len(list_1[0])  # or k = -1

    n_points = len(list_1)
    total_distance = 0
    
    for i in range(n_points):
        list1_slice = list_1[i][:k]
        list2_slice = list_2[i][:k]
        
        # Compute the distance for the i-th point
        distance = Levenshtein.distance(tuple(list1_slice), tuple(list2_slice))
        
        # Sum up the total distance
        total_distance += distance
    
    # Calculate the average distance
    avg_distance = total_distance / n_points
    
    return avg_distance





# Python3 program to make  
# an array same to another 
# using minimum number of swap 
  
# Function returns the minimum  
# number of swaps required to  
# sort the array 
# This method is taken from below post 
# https: // www.geeksforgeeks.org/ 
# minimum-number-swaps-required-sort-array/ 
def minSwapsToSort(arr, n): 
  
    # Create an array of pairs  
    # where first element is  
    # array element and second 
    # element is position of  
    # first element 
    arrPos = [[0 for x in range(2)]  
                 for y in range(n)] 
      
    for i in range(n):     
        arrPos[i][0] = arr[i] 
        arrPos[i][1] = i 
  
    # Sort the array by array  
    # element values to get right 
    # position of every element  
    # as second element of pair. 
    arrPos.sort() 
  
    # To keep track of visited  
    # elements. Initialize all  
    # elements as not visited  
    # or false. 
    vis = [False] * (n) 
  
    # Initialize result 
    ans = 0
  
    # Traverse array elements 
    for i in range(n): 
      
        # Already swapped and corrected or 
        # already present at correct pos 
        if (vis[i] or arrPos[i][1] == i): 
            continue
  
        # Find out the number of  node in 
        # this cycle and add in ans 
        cycle_size = 0
        j = i 
          
        while (not vis[j]):         
            vis[j] = 1
  
            # Move to next node 
            j = arrPos[j][1] 
            cycle_size+= 1
         
        # Update answer by  
        # adding current cycle. 
        ans += (cycle_size - 1)   
  
    # Return result 
    return ans 
  
# Method returns minimum  
# number of swap to make 
# array B same as array A 
def minSwapToMakeArraySame(a, b): 
    b_copy = copy.deepcopy(b)
    # map to store position  
    # of elements in array B 
    # we basically store  
    # element to index mapping. 
    mp = {} 
    for i in range(len(b)): 
        mp[b_copy[i]] = i 
  
    # now we're storing position  
    # of array A elements 
    # in array B. 
    for i in range(len(b)): 
        b_copy[i] = mp[a[i]] 
  
    # Returning minimum swap  
    # for sorting in modified 
    # array B as final answer 
    return minSwapsToSort(b_copy, len(b)) 



def average_min_swap_distance(list1, list2):
    total_distance = 0
    for row1, row2 in zip(list1, list2):
        total_distance += minSwapToMakeArraySame(row1, row2)
    return total_distance







# def cosine_similarity(tensor_1, tensor_2):
#     rowwise_dot_product = np.sum(tensor_1 * tensor_2, axis=1)
    
#     norm_tensor_1 = np.linalg.norm(tensor_1, axis=1)
#     norm_tensor_2 = np.linalg.norm(tensor_2, axis=1)
    
#     cosine_sim = rowwise_dot_product / (norm_tensor_1 * norm_tensor_2)
    
#     return cosine_sim

In [22]:
def get_neighbours(embedding_path: str, save: bool = False) -> List[np.ndarray]:
    assert os.path.exists(embedding_path), f"{embedding_path} does not exist"
    neighbours_fname = os.path.basename(embedding_path).replace('.pickle', '_neighbours.pickle')
    sorted_neighbours_path = os.path.join(SORTED_NEIGHBOURS_DIR_PATH, neighbours_fname)
    if os.path.exists(sorted_neighbours_path):
        with open(sorted_neighbours_path, 'rb') as f:
            sorted_neighbours = pickle.load(f)
        return sorted_neighbours
    
    with open(embedding_path, 'rb') as f:
        embeddings_pd = pickle.load(f)
    ids, embeddings = convert_pd_embeddings_to_np(embeddings_pd)
    similarity_matrix = compute_similarity(embeddings, method='cosine')
    sorted_neighbours = get_sorted_neighbours_from_similarity_matrix(similarity_matrix)
    if save:
        with open(sorted_neighbours_path, 'wb') as f:
            pickle.dump(sorted_neighbours, f)
    return sorted_neighbours


def get_metrics(embedding_paths: List[str], metric_funcs_dict: dict) -> dict:
    """
    For each neighbour pair calculate metrics from metric_funcs_dict
    """
    metrics = {}
    for metric_name, metric_func in metric_funcs_dict.items():
        metrics[metric_name] = {}
        # Iterating over each possible pair of embedding files
        for i in range(len(embedding_paths)):
            neighbours_1 = get_neighbours(embedding_paths[i])
            for j in tqdm(range(i+1, len(embedding_paths))):
                neighbours_2 = get_neighbours(embedding_paths[j])
                metric = metric_func(neighbours_1, neighbours_2)
                experiment_name_1 = os.path.basename(embedding_paths[i]).replace('.pickle', '')
                experiment_name_2 = os.path.basename(embedding_paths[j]).replace('.pickle', '')
                frozenset_experiment_names = frozenset({experiment_name_1, experiment_name_2})
                metrics[metric_name][frozenset_experiment_names] = metric
                with open('similarity_metrics.pickle', 'wb') as f:
                    pickle.dump(metrics, f)
    return metrics





In [23]:
if not os.path.exists(SORTED_NEIGHBOURS_DIR_PATH):
    os.makedirs(SORTED_NEIGHBOURS_DIR_PATH)

In [24]:
paths_to_exclude = {
    os.path.join(DATA_PATH, 'coles_avg_pool_model_embeddings.pickle')
}
embedding_paths = list(set(glob.glob(f"{DATA_PATH}/*.pickle")) - paths_to_exclude)

In [27]:
# for embedding_path in tqdm(embedding_paths):
#     with open(embedding_path, 'rb') as f:
#         neighbours_fname = os.path.basename(embedding_path).replace('.pickle', '_neighbours.pickle')
#         if os.path.exists(os.path.join(SORTED_NEIGHBOURS_DIR_PATH, neighbours_fname)):
#             continue

#         embeddings_pd = pickle.load(f)
#         ids, embeddings = convert_pd_embeddings_to_np(embeddings_pd)
#         similarity_matrix = compute_similarity(embeddings, method='cosine')
#         sorted_neighbours = get_sorted_neighbours_from_similarity_matrix(similarity_matrix)
#         sorted_neighbours_path = os.path.join(SORTED_NEIGHBOURS_DIR_PATH, neighbours_fname)
#         with open(sorted_neighbours_path, 'wb') as f:
#             pickle.dump(sorted_neighbours, f)
#         break

    

In [28]:
# len(sorted_neighbours)

In [31]:
K_OPTIONS = [20, 1000, 100]
average_difference_funcs_dict = {
    f'average_difference_top@{k}': lambda list_1, list_2: average_difference_between_neighbor_lists(list_1, list_2, k=k)
    for k in K_OPTIONS
}

LEVINSTEIN_K_OPTIONS = [10]

average_Levinstein_funcs_dict = {
    f'average_Levinstein_distance_top@{k}': lambda list_1, list_2: average_Levinstein_distance(list_1, list_2, k=k)
    for k in LEVINSTEIN_K_OPTIONS
}

metric_funcs_dict = {
    # 'average_Levinstein_distance@all': lambda list_1, list_2: average_Levinstein_distance(list_1, list_2, k=None),
    **average_difference_funcs_dict,
    **average_Levinstein_funcs_dict,
    'avg_min_swaps': average_min_swap_distance,
}

In [32]:
metrics = get_metrics(embedding_paths, metric_funcs_dict)

  4%|▍         | 5/119 [01:13<27:20, 14.39s/it]

In [ ]:
with open('similarity_metrics.pickle', 'wb') as f:
    pickle.dump(metrics, f)